# Alternative Methods for Regime Detection
### Market Regime Detection · Probabilistic ML Project

Implements and compares these non-HMM approaches:
1. **Gaussian Mixture Model (GMM)** - unsupervised, no temporal dynamics  
2. **K-Means Clustering** - hard assignment on feature space  
3. **DBSCAN** - density-based, detects outlier regimes  
4. **Agglomerative Hierarchical Clustering**  
5. **Isolation Forest** - anomaly/stress period detection  
6. **Random Forest** - supervised regime labelling  
7. **Wasserstein Clustering** - regime separation via optimal transport  

---

### References  
1. Nguyen & Nguyen (2015) - Risks 3(4) - stock selection via HMM  
2. Hikmath Technologies - *Market Regime Detection: From HMMs to Wasserstein Clustering*  
3. LSEG API Samples - *Market Regime Detection Using Statistical and ML Approaches*  
4. SSRN Paper 3406068 - *Portfolio selection with HMMs*

---

## 0. Imports & Data

In [8]:
%pip install -q scikit-learn pot hmmlearn

Note: you may need to restart the kernel to use updated packages.


In [9]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import norm, wasserstein_distance
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

from utils.data_utils import prepare_ticker_data, get_observation_sequence, time_split
from utils.viz_utils  import (plot_price_with_regimes, plot_regime_distributions,
                               plot_transition_matrix, plot_regime_stats,
                               REGIME_COLOURS, _DARK_LAYOUT)
from utils.metrics    import (regime_statistics, regime_persistence_summary,
                               regime_purity, pairwise_wasserstein, aic, bic)

print("All imports OK")

All imports OK


In [14]:
TICKER = "SPY"
df = prepare_ticker_data(TICKER, start="2024-01-01")
df_train, df_test = time_split(df, train_frac=0.80)

# Feature matrix for multi-variate clustering
FEATURE_COLS = ["Returns", "AbsReturn", "RVol_20d", "RVol_60d", "Momentum_5d"]
X_feat_train = df_train[FEATURE_COLS].values
X_feat_test  = df_test[FEATURE_COLS].values
X_1d_train   = df_train["Returns"].values
X_1d_test    = df_test["Returns"].values

scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_feat_train)
X_scaled_test  = scaler.transform(X_feat_test)

# Reference: observable vol labels for purity computation
vol_labels_train = df_train["VolRegime"].values
vol_labels_test  = df_test["VolRegime"].values

print(f"Train: {len(df_train)}  Test: {len(df_test)}")
print(f"Features: {FEATURE_COLS}")

Train: 364  Test: 92
Features: ['Returns', 'AbsReturn', 'RVol_20d', 'RVol_60d', 'Momentum_5d']


## 1. Gaussian Mixture Model (GMM)

The GMM models the return distribution as a weighted sum of Gaussians — **without** the Markov temporal structure:
$$f(r) = \sum_{j=1}^{K} \pi_j \cdot \mathcal{N}(r;\, \mu_j,\, \sigma_j^2)$$

Fitted via EM (same algorithm as Baum-Welch, but no transition matrix).

**Key difference from HMM:** state transitions are IID — the model has no memory of the previous regime.

In [15]:
# Fit GMM with K=2,3,4,5 and select via BIC
gmm_results = []
for k in range(2, 6):
    g = GaussianMixture(n_components=k, covariance_type="full",
                        random_state=42, n_init=5, max_iter=300)
    g.fit(X_1d_train.reshape(-1, 1))
    gmm_results.append({"K": k, "BIC": g.bic(X_1d_train.reshape(-1,1)),
                         "AIC": g.aic(X_1d_train.reshape(-1,1)), "model": g})

best_k = min(gmm_results, key=lambda r: r["BIC"])["K"]
print(f"BIC-optimal K = {best_k}")
for r in gmm_results:
    print(f"  K={r['K']}  BIC={r['BIC']:.1f}  AIC={r['AIC']:.1f}")

BIC-optimal K = 3
  K=2  BIC=-2341.3  AIC=-2360.8
  K=3  BIC=-2346.5  AIC=-2377.6
  K=4  BIC=-2330.8  AIC=-2373.7
  K=5  BIC=-2325.7  AIC=-2380.2


In [16]:
gmm3 = GaussianMixture(n_components=3, covariance_type="full",
                       random_state=42, n_init=10, max_iter=300)
gmm3.fit(X_1d_train.reshape(-1, 1))

labels_gmm = gmm3.predict(X_1d_train.reshape(-1, 1))

# Sort by std  (low → high vol)
stds_gmm  = np.sqrt(gmm3.covariances_.ravel())
order_gmm = np.argsort(stds_gmm)
remap_gmm = {order_gmm[k]: k for k in range(3)}
labels_gmm = np.array([remap_gmm[l] for l in labels_gmm])

df_gmm = df_train.copy()
df_gmm["GMM"] = labels_gmm

print("GMM means  :", gmm3.means_.ravel()[order_gmm].round(6))
print("GMM stds   :", stds_gmm[order_gmm].round(6))
print("GMM weights:", gmm3.weights_[order_gmm].round(4))

GMM means  : [ 0.099863  0.001993 -0.006248]
GMM stds   : [0.001    0.006358 0.017412]
GMM weights: [0.0027 0.8038 0.1935]


In [17]:
fig = plot_price_with_regimes(df_gmm, "GMM", title="SPY — GMM Regimes")
fig.show()

fig = plot_regime_distributions(df_gmm, "GMM", title="GMM — Regime Return Distributions")
fig.show()

In [18]:
# GMM soft probabilities — no Markov structure
proba_gmm = gmm3.predict_proba(X_1d_train.reshape(-1,1))[:, order_gmm]

fig = go.Figure()
state_names_3 = ["Low-Vol", "Med-Vol", "High-Vol"]
colours_3 = ["rgba(99,200,180,0.7)", "rgba(255,195,55,0.7)", "rgba(240,80,80,0.7)"]
for j in range(3):
    fig.add_trace(go.Scatter(
        x=df_train.index, y=proba_gmm[:, j],
        name=state_names_3[j], stackgroup="one",
        line=dict(color=colours_3[j]),
        fillcolor=colours_3[j],
    ))
fig.update_layout(title="GMM — Soft State Probabilities (no temporal smoothing)",
                  yaxis_title="P(state | return)", height=350, **_DARK_LAYOUT)
fig.show()

print("Purity vs vol labels:", round(regime_purity(labels_gmm, vol_labels_train), 3))

Purity vs vol labels: 0.42


## 2. K-Means Clustering

Hard cluster assignment on scaled feature space (returns, volatility, momentum).

K-Means minimises:
$$\sum_{k=1}^K \sum_{x \in C_k} \|x - \mu_k\|^2$$

No emission or transition structure is purely geometric.

In [19]:
# Elbow method + silhouette
inertias, silhouettes = [], []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled_train)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled_train, labels_k))
    print(f"K={k}  inertia={km.inertia_:.1f}  silhouette={silhouettes[-1]:.4f}")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Elbow Method", "Silhouette Score"))
fig.add_trace(go.Scatter(x=list(range(2,7)), y=inertias, mode="lines+markers",
    line=dict(color="rgba(99,200,180,0.9)")), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(2,7)), y=silhouettes, mode="lines+markers",
    line=dict(color="rgba(255,195,55,0.9)")), row=1, col=2)
fig.update_layout(height=380, title="K-Means Model Selection", **_DARK_LAYOUT)
fig.show()

K=2  inertia=1352.9  silhouette=0.4911
K=3  inertia=1064.1  silhouette=0.4845
K=4  inertia=872.4  silhouette=0.4842
K=5  inertia=710.1  silhouette=0.4639
K=6  inertia=605.5  silhouette=0.2731


In [20]:
km3 = KMeans(n_clusters=3, random_state=42, n_init=20)
labels_km = km3.fit_predict(X_scaled_train)

# Sort by average return volatility
stds_km  = [X_1d_train[labels_km == k].std() for k in range(3)]
order_km = np.argsort(stds_km)
remap_km = {order_km[k]: k for k in range(3)}
labels_km = np.array([remap_km[l] for l in labels_km])

df_km = df_train.copy()
df_km["KMeans"] = labels_km

fig = plot_price_with_regimes(df_km, "KMeans", title="SPY — K-Means Regimes")
fig.show()

fig = plot_regime_distributions(df_km, "KMeans", title="K-Means — Regime Return Distributions")
fig.show()

print("Purity vs vol labels:", round(regime_purity(labels_km, vol_labels_train), 3))

Purity vs vol labels: 0.462


## 3. DBSCAN : Density-Based Clustering

DBSCAN does not require specifying K; it finds dense clusters and labels outliers as −1.

Useful for detecting **stress / anomaly regimes** that don't fit Gaussian clusters.

In [21]:
# PCA to 2D for visualisation
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_train)

# Tune epsilon via nearest-neighbour distances
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled_train)
dists, _ = nn.kneighbors(X_scaled_train)
k_dists  = np.sort(dists[:, -1])[::-1]

fig = go.Figure(go.Scatter(y=k_dists, mode="lines",
    line=dict(color="rgba(255,195,55,0.9)")))
fig.update_layout(title="k-NN Distance Plot (elbow → ε)", xaxis_title="Points (sorted)",
                  yaxis_title="5-NN distance", height=320, **_DARK_LAYOUT)
fig.show()

In [22]:
db = DBSCAN(eps=1.5, min_samples=10)
labels_db = db.fit_predict(X_scaled_train)
n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise    = (labels_db == -1).sum()
print(f"DBSCAN clusters: {n_clusters},  noise points: {n_noise} ({n_noise/len(labels_db)*100:.1f}%)")

# Map to 0, 1, 2 (noise → 3 for plotting)
label_map = {}
cluster_ids = sorted([l for l in set(labels_db) if l != -1])
for i, c in enumerate(cluster_ids):
    label_map[c] = i
label_map[-1] = len(cluster_ids)  # noise cluster

labels_db_mapped = np.array([label_map[l] for l in labels_db])
df_db = df_train.copy()
df_db["DBSCAN"] = labels_db_mapped

fig = plot_price_with_regimes(df_db, "DBSCAN", title="SPY — DBSCAN Regimes (grey = noise/outlier)")
fig.show()

DBSCAN clusters: 1,  noise points: 27 (7.4%)


## 4. Isolation Forest : Stress Period Detection

Isolation Forest assigns **anomaly scores** to each day.
Days with anomaly score < −0.1 are labelled as "stress regime".

In [23]:
iso = IsolationForest(n_estimators=200, contamination=0.08, random_state=42)
iso.fit(X_scaled_train)
anomaly_scores = iso.decision_function(X_scaled_train)
anomaly_labels = (anomaly_scores < -0.05).astype(int)   # 1 = stress

df_iso = df_train.copy()
df_iso["Stress"] = anomaly_labels

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Anomaly Score", "SPY Price"),
                    row_heights=[0.4, 0.6])

fig.add_trace(go.Scatter(x=df_train.index, y=anomaly_scores,
    name="Anomaly Score", line=dict(color="rgba(255,195,55,0.8)", width=0.8)), row=1, col=1)
fig.add_hline(y=-0.05, line_dash="dash", line_color="rgba(240,80,80,0.8)", row=1, col=1)

fig.add_trace(go.Scatter(x=df_train.index, y=df_train["Close"],
    name="SPY Close", line=dict(color="rgba(150,180,255,0.9)", width=1.2)), row=2, col=1)

for t in df_iso.index[df_iso["Stress"] == 1]:
    fig.add_vrect(x0=t, x1=t, fillcolor="rgba(240,80,80,0.15)", line_width=0, row=2, col=1)

fig.update_layout(title="Isolation Forest — Stress Period Detection",
                  height=550, **_DARK_LAYOUT)
fig.show()

print(f"Stress days: {anomaly_labels.sum()} ({anomaly_labels.mean()*100:.1f}% of sample)")
stress_rets = X_1d_train[anomaly_labels == 1]
normal_rets = X_1d_train[anomaly_labels == 0]
print(f"Stress mean/vol: {stress_rets.mean()*252:.2%} / {stress_rets.std()*np.sqrt(252):.2%}")
print(f"Normal mean/vol: {normal_rets.mean()*252:.2%} / {normal_rets.std()*np.sqrt(252):.2%}")

Stress days: 16 (4.4% of sample)
Stress mean/vol: -71.75% / 59.95%
Normal mean/vol: 20.90% / 13.05%


## 5. Random Forest : Supervised Regime Classification

We use the **observable vol regime** (Low / Med / High) as the target label,
then train a Random Forest classifier on lagged features.

This tests whether ML can predict the *next* regime from current features.

In [24]:
from sklearn.model_selection import cross_val_score

TARGET_COL = "VolRegime"
TARGET_MAP = {"Low": 0, "Med": 1, "High": 2}

y_train_rf = df_train[TARGET_COL].map(TARGET_MAP).values

# Lag 1-day features as predictors
X_rf = X_scaled_train[1:]
y_rf = y_train_rf[1:]

rf = RandomForestClassifier(n_estimators=200, max_depth=8,
                              random_state=42, n_jobs=-1)
cv_scores = cross_val_score(rf, X_rf, y_rf, cv=5, scoring="accuracy")
print(f"CV Accuracy (5-fold): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

rf.fit(X_rf, y_rf)
labels_rf_train = rf.predict(X_scaled_train)

CV Accuracy (5-fold): 0.967 ± 0.047


In [25]:
# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
fig = go.Figure(go.Bar(x=feat_imp.values, y=feat_imp.index, orientation="h",
    marker_color="rgba(99,200,180,0.8)"))
fig.update_layout(title="Random Forest Feature Importances",
                  xaxis_title="Importance", height=350, **_DARK_LAYOUT)
fig.show()

In [26]:
# Out-of-sample accuracy
X_test_rf = scaler.transform(df_test[FEATURE_COLS].values)
y_test_rf  = df_test[TARGET_COL].map(TARGET_MAP).values

y_pred_rf = rf.predict(X_test_rf)
oos_acc   = (y_pred_rf == y_test_rf).mean()
print(f"OOS Classification Accuracy: {oos_acc:.3f}")

df_rf = df_test.copy()
df_rf["RF_Regime"] = y_pred_rf
fig = plot_price_with_regimes(df_rf, "RF_Regime",
                               title="SPY — Random Forest Regime Predictions (test set)")
fig.show()

OOS Classification Accuracy: 0.978


## 6. Wasserstein Clustering

Based on the Hikmath Technologies article, we implement regime detection by:
1. Computing **Wasserstein-1 distances** between rolling return windows  
2. Applying **K-Medoids** (PAM) clustering in Wasserstein distance space  

The **Earth Mover's Distance** between distributions $p$ and $q$:
$$W_1(p,q) = \int_{-\infty}^{\infty} |F_p(x) - F_q(x)|\, dx$$

This is more robust than Euclidean distance for comparing return distributions.

In [27]:
from scipy.stats import wasserstein_distance

WINDOW_W = 63   # ~1 quarter per window
step     = 5    # step size in days

windows = []
window_dates = []
for start in range(0, len(X_1d_train) - WINDOW_W, step):
    windows.append(X_1d_train[start:start + WINDOW_W])
    window_dates.append(df_train.index[start + WINDOW_W - 1])

n_win = len(windows)
print(f"Total windows: {n_win}")

# Pairwise Wasserstein distance matrix
print("Computing pairwise Wasserstein distances …")
D = np.zeros((n_win, n_win))
for i in range(n_win):
    for j in range(i + 1, n_win):
        d = wasserstein_distance(windows[i], windows[j])
        D[i, j] = D[j, i] = d

print(f"Distance matrix: {D.shape}, mean={D.mean():.6f}")

Total windows: 61
Computing pairwise Wasserstein distances …
Distance matrix: (61, 61), mean=0.003367


In [28]:
# Embed via MDS for visualisation
from sklearn.manifold import MDS
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42, normalized_stress=False)
coords = mds.fit_transform(D)

# K-Means in MDS space (proxy for K-Medoids)
km_w = KMeans(n_clusters=3, random_state=42, n_init=20)
wass_cluster = km_w.fit_predict(coords)

# Map cluster to date
wass_labels = np.full(len(df_train), np.nan)
for idx, date in enumerate(window_dates):
    pos = df_train.index.get_loc(date)
    wass_labels[pos] = wass_cluster[idx]

# Forward-fill for gaps
wass_series = pd.Series(wass_labels, index=df_train.index).ffill().bfill()
wass_labels_full = wass_series.values.astype(int)

df_wass = df_train.copy()
df_wass["Wass"] = wass_labels_full

fig = go.Figure(go.Scatter(
    x=coords[:, 0], y=coords[:, 1],
    mode="markers",
    marker=dict(
        color=wass_cluster,
        colorscale="Viridis",
        size=6,
        showscale=True,
    ),
    text=[str(d.date()) for d in window_dates],
    hovertemplate="Date: %{text}<br>Cluster: %{marker.color}<extra></extra>",
))
fig.update_layout(title="Wasserstein Space — MDS Embedding of Rolling Windows",
                  xaxis_title="MDS-1", yaxis_title="MDS-2",
                  height=450, **_DARK_LAYOUT)
fig.show()

In [29]:
fig = plot_price_with_regimes(df_wass, "Wass",
                              title="SPY — Wasserstein Clustering Regimes")
fig.show()

fig = plot_regime_distributions(df_wass, "Wass",
                                 title="Wasserstein Clustering — Return Distributions")
fig.show()

print("Purity vs vol labels:", round(regime_purity(wass_labels_full, vol_labels_train), 3))

Purity vs vol labels: 0.481


## 7. Head-to-Head Comparison

| Method | Has Temporal Dynamics | Probabilistic | Free Parameters | Notes |
|---|---|---|---|---|
| HMM (scratch/hmmlearn) | Yes (Markov transitions) | Yes (Posterior probs) | N²+2N | Best temporal coherence |
| GMM | No (IID) | Yes (Soft assignments) | 3K-1 | Good distributional fit, no memory |
| K-Means | No | No (Hard) | K×d | Fast, interpretable, Euclidean |
| DBSCAN | No | No (Hard) | 2 hyper | Detects outlier regimes |
| Isolation Forest | No | Yes (Score) | n_estimators | Best for anomaly/stress detection |
| Random Forest | No (Predicts next) | Yes (Class proba) | many | Supervised; needs labels |
| Wasserstein | Soft via window | No | K | Distribution-aware distance |

In [30]:
# Purity comparison across methods (vs observable vol labels)
from utils.metrics import regime_purity

methods = {
    "GMM"         : labels_gmm,
    "K-Means"     : labels_km,
    "Wass. Clust.": wass_labels_full,
    "RF (vol pred)": labels_rf_train,
}

# Import HMM labels from hmmlearn for full comparison
from hmmlearn import hmm as hmmlearn_hmm
h3 = hmmlearn_hmm.GaussianHMM(n_components=3, covariance_type="full",
                                n_iter=300, tol=1e-7, random_state=42)
h3.fit(X_1d_train.reshape(-1,1))
stds_h3 = np.sqrt(h3.covars_.ravel())
order_h3 = np.argsort(stds_h3)
remap_h3 = {order_h3[k]:k for k in range(3)}
labels_hmm3 = np.array([remap_h3[l] for l in h3.predict(X_1d_train.reshape(-1,1))])
methods["HMM (hmmlearn)"] = labels_hmm3

print("Method purity vs Vol-Regime labels:")
for name, lab in methods.items():
    print(f"  {name:20s} : {regime_purity(lab, vol_labels_train):.3f}")

Method purity vs Vol-Regime labels:
  GMM                  : 0.420
  K-Means              : 0.462
  Wass. Clust.         : 0.481
  RF (vol pred)        : 1.000
  HMM (hmmlearn)       : 0.420


In [31]:
# Persistence comparison
from utils.metrics import regime_persistence

print("\nMean regime persistence (days):")
for name, lab in methods.items():
    p = regime_persistence(lab)
    mean_p = np.mean(list(p.values()))
    print(f"  {name:20s} : {mean_p:.1f} days")


Mean regime persistence (days):
  GMM                  : 4.1 days
  K-Means              : 8.3 days
  Wass. Clust.         : 63.6 days
  RF (vol pred)        : 12.5 days
  HMM (hmmlearn)       : 2.7 days


In [32]:
# Interactive comparison chart: regime labels side-by-side
fig = make_subplots(rows=len(methods) + 1, cols=1,
                    shared_xaxes=True,
                    subplot_titles=["SPY Price"] + list(methods.keys()),
                    row_heights=[0.35] + [0.13] * len(methods))

fig.add_trace(go.Scatter(x=df_train.index, y=df_train["Close"],
    line=dict(color="rgba(150,180,255,0.9)", width=1.2), name="SPY"), row=1, col=1)

colour_seq = ["rgba(99,200,180,0.8)", "rgba(255,195,55,0.8)", "rgba(240,80,80,0.8)", "rgba(140,100,240,0.8)"]
for row_idx, (name, lab) in enumerate(methods.items(), start=2):
    fig.add_trace(go.Scatter(
        x=df_train.index, y=lab.astype(float),
        name=name,
        mode="lines",
        line=dict(color=colour_seq[(row_idx-2) % 4], width=1),
        fill="tozeroy",
        fillcolor=colour_seq[(row_idx-2) % 4].replace("0.8", "0.2"),
    ), row=row_idx, col=1)
    fig.update_yaxes(title_text=name[:12], row=row_idx, col=1,
                     tickvals=[0,1,2], ticktext=["L","M","H"])

fig.update_layout(title="Regime Detection — All Methods Side-by-Side",
                  height=900, showlegend=False, **_DARK_LAYOUT)
fig.show()

## 8. Key Takeaways

### Advantages & Shortcomings

**GMM**
- Clean probabilistic framework, soft assignments  
- Ignores time ordering, two adjacent days treated as independent  
- Gaussian assumption may not capture fat tails  

**K-Means**
- Simple, fast, interpretable  
- Euclidean distance penalises outliers; sensitive to feature scaling  
- No uncertainty estimate; hard boundaries  

**DBSCAN**
- Discovers arbitrary cluster shapes; explicit noise label  
- Sensitive to ε and min_samples; many noise points in finance  

**Isolation Forest**
- Excellent at pinpointing stress events (2008, 2020, etc.)  
- Binary output, no multi-state regime structure  

**Random Forest (supervised)**
- Highest accuracy when labels are available  
- Requires labelled training data; future labels are unknown in live trading  

**Wasserstein Clustering**
- Distribution-aware; captures shape differences, not just mean/variance  
- Computationally expensive ($O(n^2)$ distance matrix); no temporal smoothing  

**HMM**
- Principled probabilistic model with temporal dynamics (Markov property)  
- Soft state probabilities; joint sequence likelihood  
- Assumes first-order Markov (no long-range dependence)  
- Gaussian emissions may be mis-specified; can overfit